# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import pandas as pd, numpy as np
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
print(df.shape)

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
REASON_CODES = {
    "REFRESH_DECLINING": "Clicks fell vs the previous 30 days; refresh to recover them",
    "FIX_SNIPPET":       "Ranks top 10 but CTR is below peers at that position; rewrite title/meta",
    "PUSH_RANKING":      "Ranks 10-20 with real impressions; improve content to reach page one",
    "LOW_DATA":          "Too little traffic to judge; no action recommended",
}
for k, v in REASON_CODES.items():
    print(f"{k}: {v}")

REFRESH_DECLINING: Clicks fell vs the previous 30 days; refresh to recover them
FIX_SNIPPET: Ranks top 10 but CTR is below peers at that position; rewrite title/meta
PUSH_RANKING: Ranks 10-20 with real impressions; improve content to reach page one
LOW_DATA: Too little traffic to judge; no action recommended


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
import os

d = df.replace([np.inf, -np.inf], np.nan).copy()
d["ctr_30"] = d["clicks_last_30d"] / d["impressions_last_30d"].replace(0, np.nan)

# Benchmark CTR by position bucket, learned from the data itself (rows with enough impressions)
d["pos_bucket"] = pd.cut(d["avg_position"], [0, 3, 10, 20, 1000], labels=["1-3", "4-10", "11-20", "20+"])
ok = d["impressions_last_30d"] >= 100
bench_p75 = d[ok].groupby("pos_bucket", observed=True)["ctr_30"].quantile(.75)
bench_med = d[ok].groupby("pos_bucket", observed=True)["ctr_30"].median()
print("benchmark CTR (p75) by position:\n", bench_p75.round(4))

# Opportunity 1: lost clicks (needs enough prior clicks to be meaningful)
d["opp_refresh"] = np.where(d["clicks_prev_30d"] >= 10,
                            (d["clicks_prev_30d"] - d["clicks_last_30d"]).clip(lower=0), 0)

# Opportunity 2: CTR gap for top-10 content
gap = (d["pos_bucket"].map(bench_p75).astype(float) - d["ctr_30"]).clip(lower=0)
d["opp_snippet"] = np.where(d["avg_position"] <= 10, gap * d["impressions_last_30d"], 0)

# Opportunity 3: reaching page one for 10-20 (half value, since the move is uncertain)
gain = (float(bench_med["4-10"]) - d["ctr_30"]).clip(lower=0)
d["opp_push"] = np.where((d["avg_position"] > 10) & (d["avg_position"] <= 20),
                         0.5 * gain * d["impressions_last_30d"], 0)

opps = d[["opp_refresh", "opp_snippet", "opp_push"]].fillna(0)
code_map = {"opp_refresh": "REFRESH_DECLINING", "opp_snippet": "FIX_SNIPPET", "opp_push": "PUSH_RANKING"}
d["score"] = opps.max(axis=1).round(1)
d["reason_code"] = opps.idxmax(axis=1).map(code_map)

# Too little data to trust anything
low = (d["impressions_last_30d"] < 100) & (d["clicks_prev_30d"] < 10)
d.loc[low | (d["score"] == 0), "reason_code"] = "LOW_DATA"
d.loc[low | (d["score"] == 0), "score"] = 0.0

d["stale"] = d["days_since_last_update"] > 180
d["confidence"] = np.select(
    [(d["impressions_last_30d"] >= 1000) & (d["clicks_prev_30d"] >= 50),
     (d["impressions_last_30d"] >= 300)],
    ["high", "medium"], default="low")

queue = d.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
cols = ["rank", "client_id", "content_id", "reason_code", "score", "confidence", "stale",
        "clicks_prev_30d", "clicks_last_30d", "impressions_last_30d", "avg_position", "days_since_last_update"]

os.makedirs("work/outputs", exist_ok=True)
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows")
print(queue["reason_code"].value_counts())

benchmark CTR (p75) by position:
 pos_bucket
1-3      0.0080
4-10     0.0056
11-20    0.0051
20+      0.0026
Name: ctr_30, dtype: float64
wrote 30000 rows
reason_code
LOW_DATA             21241
FIX_SNIPPET           4958
PUSH_RANKING          2366
REFRESH_DECLINING     1435
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
wrong_if = {
    "REFRESH_DECLINING": "the drop is seasonal or random noise, not decay; or the page is already updated elsewhere",
    "FIX_SNIPPET": "the low CTR comes from SERP features or branded queries, not a weak title",
    "PUSH_RANKING": "the competition at positions 1-10 is too strong, so rank won't move",
}
top20 = queue.head(20).copy()
top20["what_would_make_it_wrong"] = top20["reason_code"].map(wrong_if)
top20["confidence_note"] = top20["confidence"] + " (impr30=" + top20["impressions_last_30d"].astype(int).astype(str) + \
                           ", prev clicks=" + top20["clicks_prev_30d"].astype(int).astype(str) + ")"
pd.set_option("display.max_colwidth", 80, "display.width", 250)
print(top20[["rank", "content_id", "reason_code", "score", "confidence_note", "what_would_make_it_wrong"]].to_string(index=False))

 rank           content_id       reason_code  score                        confidence_note                                                                  what_would_make_it_wrong
    1 content_8451fc6f034d       FIX_SNIPPET 1315.7 medium (impr30=168958, prev clicks=30)                 the low CTR comes from SERP features or branded queries, not a weak title
    2 content_4a6607efcb46       FIX_SNIPPET  968.4  medium (impr30=122303, prev clicks=2)                 the low CTR comes from SERP features or branded queries, not a weak title
    3 content_db5989a78dd3       FIX_SNIPPET  841.6   high (impr30=238796, prev clicks=53)                 the low CTR comes from SERP features or branded queries, not a weak title
    4 content_07e0b9af8b1a REFRESH_DECLINING  678.0  high (impr30=52440, prev clicks=1627) the drop is seasonal or random noise, not decay; or the page is already updated elsewhere
    5 content_36ff89c8214e       FIX_SNIPPET  566.5   high (impr30=106985, prev clicks=63)     

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
top = queue.head(100)

# Weak pick 1: drops no bigger than random noise (Poisson: about sqrt of prior clicks)
r = top[top["reason_code"] == "REFRESH_DECLINING"]
noise = r[(r["clicks_prev_30d"] - r["clicks_last_30d"]) < np.sqrt(r["clicks_prev_30d"])]
print("REFRESH picks whose drop is within noise:", len(noise), "of", len(r))

# Weak pick 2: low confidence in the top 100
print("low-confidence picks in top 100:", (top["confidence"] == "low").sum())

# Weak pick 3: one client dominating the queue
print("top 100 by client share:\n", top["client_id"].value_counts(normalize=True).head(5).round(2))

# Leakage check: the score may only use these columns
used = {"clicks_last_30d", "clicks_prev_30d", "impressions_last_30d", "avg_position",
        "days_since_last_update", "client_id", "content_id"}
banned_patterns = ["_90d", "tier", "trend_", "ctr", "engagement", "scroll_rate", "ai_traffic", "days_with_"]
leaks = [c for c in used if any(p in c for p in banned_patterns)]
print("banned columns used:", leaks if leaks else "none")


REFRESH picks whose drop is within noise: 0 of 22
low-confidence picks in top 100: 0
top 100 by client share:
 client_id
client_19581e27de    0.53
client_4e07408562    0.14
client_7f2253d7e2    0.11
client_f369cb89fc    0.06
client_6208ef0f77    0.03
Name: proportion, dtype: float64
banned columns used: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.